## Setup

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

## Load, spilt, and reduce the data

In [6]:
data = load_breast_cancer()
X = data.data
y = data.target

X_scaled = StandardScaler().fit_transform(X)
X_pca = PCA(n_components=2).fit_transform(X_scaled)

Xtr, Xte, ytr, yte = train_test_split(X_pca, y, test_size=0.25, stratify=y, random_state=0)

## Permutation importance on the PCA components

Train a logistic regression on the same two PCA components used byt the VQC, then use permutation importance to measure which component the classical model relies on the most.

In [7]:
clf = LogisticRegression().fit(Xtr, ytr)
print("test accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 4))

perm_importance = permutation_importance(clf, Xte, yte, n_repeats=20, random_state=0)

importance = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "importance": perm_importance.importances_mean,
    "std": perm_importance.importances_std,
})
importance

test accuracy: 0.9371


,component,importance,std
0,PC1,0.405944,0.037364
1,PC2,0.075874,0.013859


## Cross referencing with SHAP

Use the same logistic regression and PCA components, but use SHAP instead of Permutation importance to compare explainability

In [8]:
import shap

background = shap.utils.sample(Xtr, 50, random_state=0)
explainer = shap.Explainer(clf.predict_proba, background)
shap_values = explainer(Xte[:100])[..., 1]   # SHAP values for P(benign)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

shap_importance = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "mean |SHAP|": mean_abs_shap,
})
shap_importance

,component,mean |SHAP|
0,PC1,0.416100
1,PC2,0.081563
